# 03 — Extra cleaning of the dataset (beyond the paper)

Notebook 02 checked that the published `unique_m.csv` reflects the clean-up described in the paper,
and found problems that clean-up did not catch. This notebook builds a **second, cleaned dataset** that
fixes them, so that we can later train the same models on both and compare.

**The paper is reproduced on the data as published.** Nothing here changes `features.csv`; the
cleaning produces a separate `features_cleaned.csv`.

### Where the code lives

| What | File |
|---|---|
| The cleaning steps | [`src/supercon/cleaning.py`](../src/supercon/cleaning.py) — `clean`, `has_bare_oxygen`, `load_decisions` |
| One decision per problematic formula, with a reason | [`src/supercon/data/cleaning_decisions.csv`](../src/supercon/data/cleaning_decisions.csv) |
| Scale-independent composition label | `composition_key` in [`src/supercon/features.py`](../src/supercon/features.py) |
| Build `features_cleaned.csv` and `cleaning_log.csv` | [`scripts/02_build_cleaned_features.py`](../scripts/02_build_cleaned_features.py) |
| Automated checks | [`tests/test_cleaning.py`](../tests/test_cleaning.py) |

The cleaning has three steps: **(1)** corrections and removals decided formula by formula,
**(2)** removing cuprates whose oxygen content is unknown, **(3)** removing duplicates that differ only
in how the formula is scaled.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from supercon import composition_key, featurize
from supercon.cleaning import clean, has_bare_oxygen, load_decisions
from supercon.elements import ELEMENTS

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

unique_m = pd.read_csv(ROOT / "data" / "raw" / "unique_m.csv")
m = unique_m["material"]
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

## Step 1 — Decisions taken formula by formula

Every correction or removal that needs judgement is written down in
[`cleaning_decisions.csv`](../src/supercon/data/cleaning_decisions.csv): the original formula, the
action (`fix`, `remove` or `keep`), the corrected formula for fixes, and the reason. Anyone reviewing
the data can read, question or change a single line without touching the code.

In [2]:
decisions = load_decisions()
decisions["action"].value_counts()

action
remove    22
fix       10
keep       3
Name: count, dtype: int64

In [3]:
decisions[decisions["action"] == "fix"]

,action,corrected,reason
material,,,
Pr185Ce0.15Cu1O4,fix,Pr1.85Ce0.15Cu1O4,misplaced decimal (same slip as Nd185Ce0.15Cu1O4 fixed in the paper); Pr1.85Ce0.15Cu1O4 with the same Tc i...
La1.85Sr0.15Cu98Ni0.02O4,fix,La1.85Sr0.15Cu0.98Ni0.02O4,misplaced decimal; La1.85Sr0.15Cu0.98Ni0.02O4 with the same Tc is already in the data
Rb3C60C60,fix,Rb3C60,C60 written twice; C60Rb3 with the same Tc is already in the data
Nd1.88Ce0.12Cu0.98IN0.02O4,fix,Nd1.88Ce0.12Cu0.98In0.02O4,indium typed as IN (parsed as iodine + nitrogen)
Nd1.88Ce0.12Cu0.97IN0.03O4,fix,Nd1.88Ce0.12Cu0.97In0.03O4,indium typed as IN (parsed as iodine + nitrogen)
Nd1.88Ce0.12Cu0.96IN0.04O4,fix,Nd1.88Ce0.12Cu0.96In0.04O4,indium typed as IN (parsed as iodine + nitrogen)
Nd1.87Ce0.13Cu0.98IN0.02O4,fix,Nd1.87Ce0.13Cu0.98In0.02O4,indium typed as IN (parsed as iodine + nitrogen)
Nd1.85Ce0.15Cu0.99IN0.01O4,fix,Nd1.85Ce0.15Cu0.99In0.01O4,indium typed as IN (parsed as iodine + nitrogen)
Nd1.85Ce0.15Cu0.98IN0.02O4,fix,Nd1.85Ce0.15Cu0.98In0.02O4,indium typed as IN (parsed as iodine + nitrogen)


The fixes are of three kinds:

* **Misplaced decimals** (`Pr185…` → `Pr1.85…`, `Cu98` → `Cu0.98`) and `C60` written twice. Each
  corrected formula already exists in the data *with exactly the same Tc*, so these were mangled copies
  of real records; after the fix, step 3 removes them as duplicates.
* **Indium typed as `IN`**, which the parser reads as iodine + nitrogen.
* `Sm1Fe1As1F0.5O`: the oxygen coefficient was dropped. The other Sm–Fe–As–F–O records are written
  F$_x$O$_{1-x}$ (e.g. `Sm1Fe1As1F0.15O0.85`), so this is F$_{0.5}$O$_{0.5}$.

## Step 2 — Unknown oxygen content

The paper's step 7 replaced uncertain oxygen notations such as `O7-X` by a best guess (`O7`). But many
records ended up with a **bare `O`** and no number, e.g. `Y1Ba2Cu3O` for YBCO. The parser, like the
author's R code, reads a bare symbol as one atom:

In [4]:
bare = has_bare_oxygen(m)
cuprate = (unique_m["Cu"] > 0) & (unique_m["O"] > 0)
print(f"rows with a bare O: {bare.sum():,} ({bare.mean():.1%} of the data)")
print(f"  of which cuprates: {(bare & cuprate).sum():,}  ({(bare & cuprate).sum() / cuprate.sum():.0%} of all cuprates)")
print(f"  with Tc > 30 K   : {(bare & (unique_m.critical_temp > 30)).sum():,}")
m[bare & cuprate].sample(8, random_state=1).tolist()

rows with a bare O: 1,927 (9.1% of the data)
  of which cuprates: 1,901  (18% of all cuprates)
  with Tc > 30 K   : 1,663


['Tl0.5Pb0.5Ca0.8Er0.2Sr2Cu2O',
 'Nd1.32Ce0.27Sr0.41Cu1O',
 'Y2Ba4Cu7F1.2O',
 'Bi1.8Pb0.2Sr1.63La0.37Cu0.97Fe0.03O',
 'Eu1Pb1Sr2Ca3Cu4O',
 'Bi2.1Sr2.1Ca0.8Cu1.9Ni0.1S0.1O',
 'Hg1Ba2Cu1O',
 'Bi0.35In0.65Sr2Gd0.6Ca0.4Cu2O']

This is not a small effect on the features. For YBCO, one oxygen instead of seven changes the
proportions completely, and with them every weighted feature:

In [5]:
ybco = featurize(pd.DataFrame([{"Y": 1, "Ba": 2, "Cu": 3, "O": 1}, {"Y": 1, "Ba": 2, "Cu": 3, "O": 7}],
                              index=["Y1Ba2Cu3O (as recorded)", "Y1Ba2Cu3O7 (real)"]))
ybco[["wtd_mean_atomic_mass", "wtd_mean_ThermalConductivity", "wtd_entropy_Valence", "wtd_std_ElectronAffinity"]].round(2)

,wtd_mean_atomic_mass,wtd_mean_ThermalConductivity,wtd_entropy_Valence,wtd_std_ElectronAffinity
Y1Ba2Cu3O (as recorded),81.46,179.0,1.31,52.65
Y1Ba2Cu3O7 (real),51.25,96.4,1.20,49.45


We cannot know the true oxygen content of these records, and guessing it per family would add our own
assumptions. So the cleaned dataset **removes every cuprate written with a bare `O`** (a rule, applied
in `clean`).

For **non-cuprates**, a bare `O` is sometimes correct: the "1111" iron pnictides such as `SmFeAsO`
really have one oxygen. So each of those 22 formulas has an explicit decision in the CSV instead of a
rule, and `clean` raises an error if a new one appears without a decision:

In [6]:
non_cuprate = decisions.loc[decisions.index.isin(m[bare & (unique_m.Cu == 0)]), ["action", "reason"]]
non_cuprate.sort_values("action")

,action,reason
material,,
Sm1Fe1As1F0.5O,fix,oxygen coefficient dropped; sibling records are written FxO(1-x)
La0.85Sr0.15Fe1As1O,keep,1111 iron pnictide: O1 is the correct stoichiometry
Pr1Fe1As1O,keep,1111 iron pnictide: O1 is the correct stoichiometry
Sm1Fe1As1O,keep,1111 iron pnictide: O1 is the correct stoichiometry
Ca6Sc2Ti3Fe2As2O,remove,oxygen content unknown (perovskite-block iron arsenide)
Ca5Sc2Ti2Fe2As2O,remove,oxygen content unknown (perovskite-block iron arsenide)
Ca4Sc2.01Ti0.99Fe2As2O,remove,oxygen content unknown (perovskite-block iron arsenide)
Fe1.08Te0.55Se0.45O,remove,oxygen content unknown (O-incorporated FeTe1-xSex)
Fe1.08Te1O,remove,oxygen content unknown (O-incorporated FeTe)


Four cuprates also carry an explicit `O1` that is clearly a placeholder (e.g. `Hg1Ba2Ca3Cu4O1` at
115 K); they are removed by decision. `Bi1Cu0.9S1O1` (BiCuSO) is kept: it really has one oxygen.

## Step 3 — Duplicates across scales

The author removed a row only when the element counts *and* Tc matched exactly. The same compound
written at another scale survived, although its 81 features are identical, because the features depend
only on the proportions. `composition_key` gives every composition a scale-independent label:

In [7]:
example = pd.DataFrame([{"Si": 1, "V": 3}, {"Si": 0.25, "V": 0.75}], index=["Si1V3", "Si0.25V0.75"])
composition_key(example)

Si1V3          Si0.25V0.75
Si0.25V0.75    Si0.25V0.75
Name: composition, dtype: str

Rows with the same composition key **and** the same Tc are duplicates and are dropped. Rows with the
same composition but a *different* Tc (repeated measurements) are kept: they are real data, and the
modelling step decides how to use them (the key serves as the group for train/test splits).

## Running the whole cleaning

`clean` applies the three steps and returns the cleaned table plus a log with one row per action.
[`scripts/02_build_cleaned_features.py`](../scripts/02_build_cleaned_features.py) runs the same
function and saves both to `data/processed/`.

In [8]:
cleaned, log = clean(unique_m)
print(f"{len(unique_m):,} rows -> {len(cleaned):,} rows")
log.groupby("step").size()

21,263 rows -> 19,264 rows


step
duplicate      73
fix            13
remove       1926
dtype: int64

In [9]:
log[log["row"].isin([4725, 17614, 2125])]

,row,material,step,new_material,reason
439,2125,Rb3C60,duplicate,,same composition (C0.9523809524Rb0.0476190476) and Tc as another row
440,2125,Rb3C60C60,fix,Rb3C60,C60 written twice; C60Rb3 with the same Tc is already in the data
916,4725,Pr1.85Ce0.15Cu1O4,duplicate,,same composition (O0.5714285714Cu0.1428571429Ce0.0214285714Pr0.2642857143) and Tc as another row
917,4725,Pr185Ce0.15Cu1O4,fix,Pr1.85Ce0.15Cu1O4,misplaced decimal (same slip as Nd185Ce0.15Cu1O4 fixed in the paper); Pr1.85Ce0.15Cu1O4 with the same Tc i...
1937,17614,La1.85Sr0.15Cu0.98Ni0.02O4,duplicate,,same composition (O0.5714285714Ni0.0028571429Cu0.14Sr0.0214285714La0.2642857143) and Tc as another row
1938,17614,La1.85Sr0.15Cu98Ni0.02O4,fix,La1.85Sr0.15Cu0.98Ni0.02O4,misplaced decimal; La1.85Sr0.15Cu0.98Ni0.02O4 with the same Tc is already in the data


The log shows, for instance, that `Pr185Ce0.15Cu1O4` (row 4725) was first corrected and then dropped
as a duplicate of the existing `Pr1.85Ce0.15Cu1O4` record with the same Tc.

## What changes between the two datasets

In [10]:
def summary(df):
    tc = df["critical_temp"]
    cup = (df["Cu"] > 0) & (df["O"] > 0)
    return pd.Series({
        "rows": f"{len(df):,}",
        "distinct compositions": f"{composition_key(df[ELEMENTS]).nunique():,}",
        "cuprates": f"{cup.sum():,}",
        "share Tc > 30 K": round((tc > 30).mean(), 3),
        "Tc median": tc.median(),
        "Tc mean": round(tc.mean(), 1),
    })

pd.DataFrame({"published": summary(unique_m), "cleaned": summary(cleaned)}, dtype=object)

,published,cleaned
rows,"21,263","19,264"
distinct compositions,"15,164","13,701"
cuprates,"10,532","8,621"
share Tc > 30 K,0.41,0.366
Tc median,20.0,16.5
Tc mean,34.4,30.9


### What this means for comparing the models

* The cleaning removes about 1,900 high-Tc cuprates, so the two datasets have **different Tc
  distributions**. An RMSE computed on each dataset's own test set is therefore not a fair comparison:
  the cleaned test set is, on average, easier.
* The fair comparison is on the **same test rows**: split by composition, then score both models only on
  test compositions present in both datasets. The removed rows can additionally be used to see how the
  model trained on the published data behaves on records with unknown oxygen.
* Both `features.csv` and `features_cleaned.csv` carry the `composition` column, so the same grouped
  split can be applied to both.